In [1]:
!pip install transformers datasets accelerate peft bitsandbytes sentence-transformers bert-score pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.3 MB/s eta 0:00:00


In [2]:
!pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 465.5/465.5 kB 26.5 MB/s eta 0:00:00


In [3]:

import pandas as pd
import json
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model
from trl import SFTTrainer
from sentence_transformers import SentenceTransformer, util
from bert_score import score

In [4]:
# 1. กำหนดชื่อไฟล์ข้อมูล
TRAIN_DATA_FILE = "/content/iot_train_set.json" # << แก้เป็นชื่อไฟล์ Train JSON ของคุณ
TEST_DATA_FILE = "/content/iot_test_set.json"   # << แก้เป็นชื่อไฟล์ Test JSON ของคุณ



In [5]:
MODEL_ID = "openthaigpt/openthaigpt1.5-7b-instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [6]:
tokenizer.chat_template = (
    "{% for message in messages %}"
    "{% if message['role'] == 'user' %}"
    "USER: {{ message['content'] }}\n"
    "{% elif message['role'] == 'assistant' %}"
    " {{ message['content'] }}{% endif %}"
    "{% endfor %}"
)

In [7]:
def format_chat(sample):
    messages = [
        {"role": "user", "content": sample["input"]},
        {"role": "assistant", "content": sample["output"]}
    ]

    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": formatted_text.strip() + tokenizer.eos_token}

def load_and_format_data(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    dataset = Dataset.from_list(data)
    return dataset.map(format_chat)

In [8]:
train_dataset = load_and_format_data(TRAIN_DATA_FILE)
eval_dataset = load_and_format_data(TEST_DATA_FILE)

print(f"ขนาดชุดข้อมูลฝึก: {len(train_dataset)}")
print(f"ตัวอย่างข้อความที่ถูกจัดรูปแบบ: \n{train_dataset[0]['text'][:300]}")

Map:   0%|          | 0/10407 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

ขนาดชุดข้อมูลฝึก: 10407
ตัวอย่างข้อความที่ถูกจัดรูปแบบ: 
USER: มีทุนการศึกษาให้ไหมครับ
 มีทุนการศึกษาจากทั้งคณะวิศวกรรมศาสตร์และของสถาบันฯ ครับ นักศึกษาสามารถติดตามประกาศและยื่นสมัครได้ตามเงื่อนไขที่กำหนด<|im_end|>


In [2]:
pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 43.8 MB/s eta 0:00:00


In [10]:
# 1. ตั้งค่า Quantization (BitsAndBytes)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# 2. โหลดโมเดล (ต้องใช้ device_map="auto" และ torch_dtype)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True # WangChanGLM อาจต้องใช้
)

# 3. เตรียมโมเดลสำหรับ K-bit Training และ LoRA
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

# 4. ตั้งค่า LoRA Config (R=64 เพื่อความแม่นยำภาษาไทยที่สูงขึ้น)
peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.15,
    r= 32,
    bias="none",
    task_type="CAUSAL_LM",
    # Target Modules ของ WangChanGLM มักจะเป็น Key/Query/Value
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

# 5. ติดตั้ง LoRA Adapter
model = get_peft_model(model, peft_config)

print("✅ ติดตั้ง LoRA Adapter สำเร็จ")
model.print_trainable_parameters()

config.json:   0%|          | 0.00/692 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.88G [00:00<?, ?B/s]

...

model-00002-of-00004.safetensors:   0%|          | 0.00/4.93G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.33G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

✅ ติดตั้ง LoRA Adapter สำเร็จ
trainable params: 80,740,352 || all params: 7,696,356,864 || trainable%: 1.0491


In [11]:
# 1. ตั้งค่า Training Arguments
OUTPUT_DIR = "./wangchanglm_iot_results"
training_args = TrainingArguments(
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    num_train_epochs=10, # เพิ่ม Epoch ให้มีเวลาเรียนรู้มากขึ้น

    # ❗ ใช้ LR ที่เคยได้ผลดี หรือต่ำกว่า เพื่อป้องกัน Overfit
    learning_rate=5e-5,  # หรือ 1e-5 ถ้าต้องการความละเอียดสูงสุด

    fp16=False,
    bf16=True,
    logging_steps=10,
    output_dir=OUTPUT_DIR,

    # ใช้ Paged Optimizer เพื่อความเสถียร
    optim="paged_adamw_8bit",

    # ❗ ใช้ Cosine Scheduler เพื่อควบคุม LR
    lr_scheduler_type="cosine",

    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    load_best_model_at_end=True,
    report_to="none",
)

In [ ]:
!pip show trl


Name: trl
Version: 0.25.1
Summary: Train transformer language models with reinforcement learning.
Home-page: https://github.com/huggingface/trl
Author: 
Author-email: Leandro von Werra <leandro.vonwerra@gmail.com>
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: accelerate, datasets, transformers
Required-by: 


In [12]:
from trl import SFTTrainer

In [13]:
tokenizer.model_max_length = 16384
tokenizer.padding_side = "right"
tokenizer.truncation_side = "right"


In [14]:
def format_example(example):
    return example["text"]


In [15]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=peft_config,
    formatting_func=format_example,

)

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:282: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Applying formatting function to train dataset:   0%|          | 0/10407 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/10407 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/10407 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/10407 [00:00<?, ? examples/s]

Applying formatting function to eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

In [16]:
trainer.tokenizer = tokenizer  # ต้อง assign เอง

Trainer.tokenizer is now deprecated. You should use `Trainer.processing_class = processing_class` instead.


In [17]:
# 3. เริ่มการฝึก
print("--- เริ่มต้นการ Fine-tuning WangChanGLM ---")
trainer_stats = trainer.train()



The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


--- เริ่มต้นการ Fine-tuning WangChanGLM ---


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
100,1.023300,1.013791,1.146582,39836.000000,0.750782
200,0.828700,0.840644,0.994407,78945.000000,0.784241
300,0.736100,0.743873,0.887667,118689.000000,0.806980
400,0.614100,0.670794,0.814513,158100.000000,0.823702
500,0.622700,0.615793,0.740900,198276.000000,0.834444
600,0.557300,0.573888,0.701455,237754.000000,0.846206
700,0.598100,0.540845,0.681223,278014.000000,0.854373
800,0.518900,0.502239,0.625342,317816.000000,0.864448
900,0.471900,0.465218,0.602544,358073.000000,0.875133
1000,0.528400,0.446528,0.569359,397267.000000,0.876879


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

KeyboardInterrupt: 

In [18]:
# 4. บันทึกโมเดลที่ฝึกเสร็จแล้ว (LoRA Adapters)
trainer.save_model(f"./model_checkpoint")

In [20]:
from google.colab import drive

# โค้ดนี้จะเปิดหน้าต่างให้คุณยืนยันการเชื่อมต่อและใส่รหัส PIN/เลือกบัญชี
print("--- เริ่มการเชื่อมต่อ Google Drive ---")
drive.mount('/content/drive')
print("--- เชื่อมต่อ Google Drive สำเร็จ ---")

--- เริ่มการเชื่อมต่อ Google Drive ---
Mounted at /content/drive
--- เชื่อมต่อ Google Drive สำเร็จ ---


In [22]:
# กำหนดชื่อโฟลเดอร์ผลลัพธ์ (OUTPUT_DIR) และชื่อไฟล์ ZIP ที่ต้องการ
OUTPUT_DIR = "./content/model_checkpoint"
ZIP_FILE_NAME = "wangchanglm_iot_finetuned_best_0821.zip"

# กำหนดปลายทางใน Google Drive
DRIVE_PATH = f"/content/drive/MyDrive/{ZIP_FILE_NAME}"

print(f"--- กำลังบีบอัดและบันทึกไฟล์ไปที่: {DRIVE_PATH} ---")

# ใช้คำสั่ง zip -r เพื่อบีบอัดโฟลเดอร์ทั้งหมดและบันทึกไปยัง Drive
!zip -r {DRIVE_PATH} {OUTPUT_DIR}

print("--- บันทึกไฟล์ ZIP สำเร็จ! ---")

--- กำลังบีบอัดและบันทึกไฟล์ไปที่: /content/drive/MyDrive/wangchanglm_iot_finetuned_best_0821.zip ---
	zip warning: name not matched: ./content/model_checkpoint

zip error: Nothing to do! (try: zip -r /content/drive/MyDrive/wangchanglm_iot_finetuned_best_0821.zip . -i ./content/model_checkpoint)
--- บันทึกไฟล์ ZIP สำเร็จ! ---


In [23]:
import os

# 1. ตรวจสอบว่าโฟลเดอร์ผลลัพธ์มีอยู่จริง
OUTPUT_DIR = "/content/model_checkpoint"
if not os.path.isdir(OUTPUT_DIR):
    print(f"🚨 ข้อผิดพลาด: ไม่พบโฟลเดอร์ผลลัพธ์ที่: {OUTPUT_DIR}")
    print("กรุณาตรวจสอบชื่อโฟลเดอร์")
else:
    # 2. กำหนดปลายทางใน Google Drive
    ZIP_FILE_NAME = "wangchanglm_iot_finetuned_best_0821.zip"
    DRIVE_PATH = f"/content/drive/MyDrive/{ZIP_FILE_NAME}"

    print(f"--- กำลังบีบอัดโฟลเดอร์ {OUTPUT_DIR} และบันทึกไปที่: {DRIVE_PATH} ---")

    # คำสั่งที่ถูกต้อง: บีบอัดโฟลเดอร์ OUTPUT_DIR ทั้งหมด
    # !zip -r คือบีบอัดแบบ Recursive (รวมซับโฟลเดอร์ทั้งหมด)
    !zip -r {DRIVE_PATH} {OUTPUT_DIR}

    print("--- บันทึกไฟล์ ZIP สำเร็จ! ---")

--- กำลังบีบอัดโฟลเดอร์ /content/model_checkpoint และบันทึกไปที่: /content/drive/MyDrive/wangchanglm_iot_finetuned_best_0821.zip ---
updating: content/model_checkpoint/ (stored 0%)
updating: content/model_checkpoint/training_args.bin (deflated 53%)
updating: content/model_checkpoint/tokenizer_config.json (deflated 89%)
updating: content/model_checkpoint/adapter_config.json (deflated 59%)
updating: content/model_checkpoint/adapter_model.safetensors (deflated 7%)
updating: content/model_checkpoint/README.md (deflated 66%)
updating: content/model_checkpoint/special_tokens_map.json (deflated 69%)
updating: content/model_checkpoint/tokenizer.json (deflated 81%)
updating: content/model_checkpoint/chat_template.jinja (deflated 42%)
  adding: content/model_checkpoint/added_tokens.json (deflated 67%)
  adding: content/model_checkpoint/vocab.json (deflated 61%)
  adding: content/model_checkpoint/merges.txt (deflated 57%)
--- บันทึกไฟล์ ZIP สำเร็จ! ---


In [32]:
import pandas as pd
import json
from tqdm import tqdm
import torch

# 1. โหลดไฟล์ iot_test_set.json
TEST_FILE_PATH = "iot_test_set.json"
try:
    with open(TEST_FILE_PATH, 'r', encoding='utf-8') as f:
        data = json.load(f)
    df = pd.DataFrame(data)
except FileNotFoundError:
    print(f"🚨 ไม่พบไฟล์: {TEST_FILE_PATH} กรุณาตรวจสอบชื่อไฟล์")
    raise

# 2. เตรียมตัวแปร (กำหนดค่าให้ model_outputs, device)
model_outputs = []
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [35]:
print("--- เริ่มการทำนายคำตอบ (Inference) ด้วยโมเดล ---")

# Initialize the 'model_output' column with empty strings or NaN if preferred
df['model_output'] = ""

for index, row in tqdm(df.iterrows(), total=len(df)):
    prompt = row['input']

    # 3.1 Tokenize Input และส่งเข้า device
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    # ❗ ลบ attention_mask ออก (เพื่อความสะอาด)
    if 'attention_mask' in inputs:
        del inputs['attention_mask']

    with torch.no_grad():
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):

            # ★● แก้ไข: ปิด use_cache เพื่อหลีกเลี่ยง ValueError
            output_ids = model.generate(
                input_ids=inputs['input_ids'],
                max_new_tokens=512,
                temperature=0.7,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
                use_cache=False # ★● การแก้ไขขั้นสุดท้าย
            )

        # 3.4 Decode Output
        model_answer = tokenizer.decode(output_ids[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

    # Assign model_answer directly to the DataFrame within the loop
    df.loc[index, 'model_output'] = model_answer

print("--- การทำนายผลเสร็จสิ้น ---")
print(f"ตัวอย่างผลลัพธ์ (คอลัมน์ input, output, model_output):\n{df[['input', 'output', 'model_output']].head()}")

--- เริ่มการทำนายคำตอบ (Inference) ด้วยโมเดล ---


  0%|          | 0/200 [00:00<?, ?it/s]


NameError: name 'model' is not defined

In [27]:
import os
import json

# กำหนดชื่อไฟล์ output
EVAL_OUTPUT_FILE = "iot_evaluation_results_final.json"

# 1. บันทึก DataFrame ที่มีคอลัมน์ model_output ลงในไฟล์ JSON
# orient='records' ทำให้ output เป็น list of objects ซึ่งเหมาะสำหรับการประเมินผล
df.to_json(EVAL_OUTPUT_FILE,
           orient='records',
           indent=4,
           force_ascii=False) # force_ascii=False เพื่อให้ภาษาไทยไม่เพี้ยน

print(f"\n✅ บันทึกไฟล์ผลลัพธ์สำหรับ Evaluation สำเร็จที่: {EVAL_OUTPUT_FILE}")

# 2. อ่านไฟล์ JSON ที่บันทึกไว้และแสดงตัวอย่าง 5 แถวแรก
print("\n--- ตัวอย่างเนื้อหา 5 แถวแรกในไฟล์ JSON ที่บันทึกแล้ว ---")

with open(EVAL_OUTPUT_FILE, 'r', encoding='utf-8') as f:
    results = json.load(f)

# แสดงผลลัพธ์ 5 รายการแรก
print(json.dumps(results[:5], indent=4, ensure_ascii=False))


✅ บันทึกไฟล์ผลลัพธ์สำหรับ Evaluation สำเร็จที่: iot_evaluation_results_final.json

--- ตัวอย่างเนื้อหา 5 แถวแรกในไฟล์ JSON ที่บันทึกแล้ว ---
[
    {
        "input": "หลักสูตรวิศวกรรมระบบไอโอทีและสารสนเทศเรียนเกี่ยวกับอะไรบ้างครับ?",
        "output": "เราสอนครอบคลุมตั้งแต่ฮาร์ดแวร์ เช่น เซ็นเซอร์และไมโครคอนโทรลเลอร์ ไปจนถึงซอฟต์แวร์ การจัดการข้อมูลบนคลาวด์ และการพัฒนาแอปพลิเคชันสำหรับอุปกรณ์ IoT ครับ",
        "model_output": ""
    },
    {
        "input": "บัณฑิตไอโอทีทำงานในโรงงานอุตสาหกรรมได้ไหมครับ",
        "output": "ทำงานได้หลากหลายตำแหน่งในสายโรงงานครับ เช่น IIoT Engineer, Automation Support, หรือตำแหน่งที่เกี่ยวข้องกับการทำ Machine Monitoring และ Energy Management System",
        "model_output": ""
    },
    {
        "input": "สาขาไอโอทีมีเรียนเกี่ยวกับความปลอดภัยของข้อมูลมั้ย",
        "output": "มีรายวิชาที่สอนเกี่ยวกับความปลอดภัยของระบบ (Cybersecurity) ครับ ซึ่งเป็นสิ่งสำคัญมากสำหรับระบบ IoT เพื่อป้องกันการโจมตีและการเข้าถึงข้อมูลโดยไม่ได้รับอนุญาต",
        "model_o

In [ ]:
import re

print("--- เริ่มทำความสะอาดคอลัมน์ 'model_output' ---")

# กำหนดรูปแบบ Prefix ที่ต้องการลบ
prefixes_to_remove = [
    r'\?\s*ASSISTANT:\s*', # เช่น "? ASSISTANT: "
    r'ครับ\s*ASSISTANT:\s*', # เช่น "ครับ ASSISTANT: "
    r'ASSISTANT:\s*', # เช่น "ASSISTANT: "
    r'ครับ\s*' # ลบคำว่า "ครับ" ที่อาจจะติดมาตอนต้น
]

# รวม Regex ทั้งหมด
regex_pattern = '|'.join(prefixes_to_remove)

# 1. ทำความสะอาดโดยใช้ Regular Expression
df['model_output'] = df['model_output'].apply(
    lambda x: re.sub(regex_pattern, '', str(x), flags=re.IGNORECASE).strip()
)

# 2. ตรวจสอบและรายงานจำนวนคำตอบที่เป็นค่าว่าง
empty_count = (df['model_output'] == '').sum()
print(f"🔹 พบคำตอบว่าง (Empty String) ทั้งหมด: {empty_count} รายการ")

print("--- ทำความสะอาดเสร็จสิ้น ---")

--- เริ่มทำความสะอาดคอลัมน์ 'model_output' ---
🔹 พบคำตอบว่าง (Empty String) ทั้งหมด: 53 รายการ
--- ทำความสะอาดเสร็จสิ้น ---


In [15]:
import re
import pandas as pd
import json
from bert_score import score
from sentence_transformers import SentenceTransformer, util

# ❗❗ กำหนดชื่อไฟล์ JSON ที่คุณบันทึกไว้
GENERATED_JSON_FILE = "/content/iot_results.json"
BERT_MODEL = 'xlm-roberta-base'
TARGET_REF_COLUMN = 'output'

# 1. โหลดข้อมูล (เพื่อให้แน่ใจว่าเรากำลังทำงานกับ DataFrame ที่ถูกต้อง)
try:
    df = pd.read_json(GENERATED_JSON_FILE, orient='records')
except Exception as e:
    print(f"❗ ข้อผิดพลาด: ไม่พบไฟล์ '{GENERATED_JSON_FILE}'")
    exit()

print("--- เริ่มทำความสะอาดคอลัมน์ 'model_output' ---")
# 2. โค้ดสำหรับทำความสะอาด
prefixes_to_remove = [
    r'\?\s*ASSISTANT:\s*',
    r'ครับ\s*ASSISTANT:\s*',
    r'คะ\s*ASSISTANT:\s*', # เพิ่ม "คะ" เข้ามา
    r'ASSISTANT:\s*',
    r'ครับ\s*',
    r'คะ\s*'
]
regex_pattern = '|'.join(prefixes_to_remove)

df['model_output'] = df['model_output'].apply(
    lambda x: re.sub(regex_pattern, '', str(x), flags=re.IGNORECASE).strip()
)
# แทนที่ Empty String ด้วยข้อความสั้นๆ เพื่อป้องกัน Metric Error
df['model_output'] = df['model_output'].replace('', 'ไม่มีคำตอบ')

# 3. เตรียมข้อมูลที่สะอาดแล้ว
candidates = df['model_output'].tolist()
references = df[TARGET_REF_COLUMN].tolist()

# 4. คำนวณ Metric ใหม่
print(f"--- เริ่มต้นคำนวณ BERTScore ใหม่ (หลังทำความสะอาด) ---")

# --- Metric 1: BERTScore ---
P, R, F1 = score(candidates, references, lang="th", model_type="xlm-roberta-base", verbose=False)
df["bert_score_f1_cleaned"] = F1.numpy()

# --- Metric 2: Sentence-BERT ---
print("     🔹 Calculating Cosine Similarity...")
sbert_model = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')
embeddings1 = sbert_model.encode(candidates, convert_to_tensor=True)
embeddings2 = sbert_model.encode(references, convert_to_tensor=True)

cosine_scores = util.cos_sim(embeddings1, embeddings2)
df["cosine_sim_cleaned"] = [cosine_scores[i][i].item() for i in range(len(candidates))]

print("\n--- สรุปผลลัพธ์เชิงความหมายใหม่ (Cleaned Results) ---")
print(f"BERTScore F1 (Mean):  {df['bert_score_f1_cleaned'].mean():.4f}")
print(f"Cosine Sim (Mean):    {df['cosine_sim_cleaned'].mean():.4f}")

--- เริ่มทำความสะอาดคอลัมน์ 'model_output' ---
--- เริ่มต้นคำนวณ BERTScore ใหม่ (หลังทำความสะอาด) ---
     🔹 Calculating Cosine Similarity...

--- สรุปผลลัพธ์เชิงความหมายใหม่ (Cleaned Results) ---
BERTScore F1 (Mean):  0.8976
Cosine Sim (Mean):    0.8126


In [28]:
# ติดตั้ง 7z ใน Colab/Linux
!sudo apt-get install p7zip-full

# กำหนดชื่อไฟล์ ZIP ที่ถูกต้องจาก Google Drive
CORRECT_ZIP_PATH = "/content/drive/MyDrive/wangchanglm_iot_finetuned_best_0821.zip"
# กำหนดโฟลเดอร์ปลายทางสำหรับการแตกไฟล์
EXTRACT_DIR = "/content/lora_adapter_folder"

# ลบโฟลเดอร์เก่าถ้ามีอยู่
!rm -rf {EXTRACT_DIR}

# สร้างโฟลเดอร์ใหม่
!mkdir -p {EXTRACT_DIR}

# แตกไฟล์ด้วย 7z
print(f"--- กำลังแตกไฟล์ {CORRECT_ZIP_PATH} ไปยัง {EXTRACT_DIR} ---")
!7z x "{CORRECT_ZIP_PATH}" -o{EXTRACT_DIR}
print("--- แตกไฟล์เสร็จสิ้น ---")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
p7zip-full is already the newest version (16.02+dfsg-8).
0 upgraded, 0 newly installed, 0 to remove and 41 not upgraded.
--- กำลังแตกไฟล์ /content/drive/MyDrive/wangchanglm_iot_finetuned_best_0821.zip ไปยัง /content/lora_adapter_folder ---

7-Zip [64] 16.02 : Copyright (c) 1999-2016 Igor Pavlov : 2016-05-21
p7zip Version 16.02 (locale=en_US.UTF-8,Utf16=on,HugeFiles=on,64 bits,8 CPUs Intel(R) Xeon(R) CPU @ 2.30GHz (306F0),ASM,AES-NI)

Scanning the drive for archives:
  0M Scan /content/drive/MyDrive/                                 1 file, 305615233 bytes (292 MiB)

Extracting archive: /content/drive/MyDrive/wangchanglm_iot_finetuned_best_0821.zip
--
Path = /content/drive/MyDrive/wangchanglm_iot_finetuned_best_0821.zip
Type = zip
Physical Size = 305615233

  0%      5% 4 - content/model_checkpoint/adapter_model.safeten

In [26]:
from google.colab import drive

# รันโค้ดนี้เพื่อ Mount Drive
print("กำลัง Mount Google Drive...")
drive.mount('/content/drive')
print("✅ Mount Drive สำเร็จ")

กำลัง Mount Google Drive...
Mounted at /content/drive
✅ Mount Drive สำเร็จ


In [1]:
import torch
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from tqdm.auto import tqdm # ใช้ tqdm.auto เพื่อความเข้ากันได้

# --- 🎯 การตั้งค่า ---
MODEL_ID = "openthaigpt/openthaigpt1.5-7b-instruct"
# ❗❗❗ แก้ไข LORA_PATH ให้ตรงกับโฟลเดอร์ที่แตกไฟล์ออกมา ❗❗❗
LORA_PATH = "/content/lora_adapter_folder/content/model_checkpoint"

# --- 1. ตั้งค่า 4-bit Quantization ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# --- 2. โหลด Base Model ---
print("กำลังโหลด Base Model ในรูปแบบ 4-bit...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

# --- 3. โหลด Tokenizer ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
device = base_model.device # กำหนด device ที่โมเดลโหลดอยู่

# --- 4. โหลดและผสาน LoRA Adapters ---
print(f"กำลังโหลดและผสาน LoRA Adapters จาก {LORA_PATH}...")
try:
    model = PeftModel.from_pretrained(
        base_model,
        LORA_PATH,
    )
    print("✅ โหลดโมเดล QLoRA ที่ Fine-tune แล้วสำเร็จ")
except Exception as e:
    print(f"❌ ERROR: ไม่สามารถโหลด LoRA Adapter ได้: {e}")
    print("โปรดตรวจสอบ LORA_PATH อีกครั้งว่าชี้ไปยังโฟลเดอร์ที่มีไฟล์ 'adapter_config.json' หรือไม่")
    # ใช้ base_model แทนในกรณีที่โหลด LoRA ไม่ได้
    model = base_model

กำลังโหลด Base Model ในรูปแบบ 4-bit...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

กำลังโหลดและผสาน LoRA Adapters จาก /content/lora_adapter_folder/content/model_checkpoint...
✅ โหลดโมเดล QLoRA ที่ Fine-tune แล้วสำเร็จ


In [13]:
# --- 5. โหลดข้อมูล JSON เข้าสู่ DataFrame ---
# (ส่วนนี้รันสำเร็จแล้ว)

# --- 6. เริ่มการทำนายคำตอบ (Inference) ด้วยโมเดล ---
print("\n--- เริ่มการทำนายคำตอบ (Inference) ด้วยโมเดล ---")

# Initialize the 'model_output' column
df['model_output'] = ""

# 🎯 กำหนด System Role ใหม่ที่ต้องการ
SYSTEM_PROMPT = "คุณคือผู้ช่วยตอบคำถามเกี่ยวกับไอโอทีและจะตอบคำถามเกี่ยวกับไอโอทีเท่านั้น"

# ใช้ tqdm.auto.tqdm ในการวนซ้ำ
from tqdm.auto import tqdm

for index, row in tqdm(df.iterrows(), total=len(df)):
    # 6.1 เตรียม Prompt และ System Role ในรูปแบบ Message List
    user_prompt = row['input']

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt}
    ]
    formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # 6.2 Tokenize Input และส่งเข้า device
    # ใช้ formatted_prompt เป็น Input หลัก
    inputs = tokenizer([formatted_prompt], return_tensors="pt").to(device)

    # ❗❗❗ ตอนนี้ไม่ต้องลบ attention_mask แล้ว ❗❗❗

    with torch.no_grad():
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):

            # 6.3 Generate Output
            # ส่ง **inputs เพื่อให้ attention_mask ถูกใช้โดยอัตโนมัติ
            output_ids = model.generate(
                **inputs,
                max_new_tokens=256, # ควบคุมความยาว
                temperature=0.7,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
                use_cache=False
            )

        # 6.4 Decode Output
        # ตัดส่วน input prompt (รวม System Role) ออกจาก output
        model_answer = tokenizer.decode(output_ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

    # 6.5 บันทึกผลลัพธ์
    df.loc[index, 'model_output'] = model_answer

print("\n--- การทำนายผลเสร็จสิ้น ---")

# --- 7. แสดงผลลัพธ์ 5 ตัวอย่างแรก ---
print(f"ตัวอย่างผลลัพธ์ (คอลัมน์ input, output, model_output):\n{df[['input', 'output', 'model_output']].head()}")


--- เริ่มการทำนายคำตอบ (Inference) ด้วยโมเดล ---


  0%|          | 0/200 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [14]:
# 1. กรองเอาเฉพาะแถวที่มีคำตอบแล้ว (ที่ไม่ใช่ค่าว่าง)
df_partial = df[df['model_output'] != ""].copy()

print(f"✅ คุณมีข้อมูลที่ทำเสร็จแล้วจำนวน: {len(df_partial)} แถว")

# 2. บันทึกข้อมูลส่วนนี้ไปทดสอบ BERT Score
if len(df_partial) > 0:
    OUTPUT_FILE = "iot_results.json"
    df_partial.to_json(OUTPUT_FILE, orient="records", indent=4, force_ascii=False)
    print(f"💾 บันทึกไฟล์ทดสอบเรียบร้อย: {OUTPUT_FILE}")
    print("👉 คุณสามารถนำไฟล์นี้ไปเขียนโค้ดคำนวณ BERT Score ได้เลยครับ")
else:
    print("⚠️ ยังไม่มีข้อมูลที่ทำเสร็จเลยครับ")

✅ คุณมีข้อมูลที่ทำเสร็จแล้วจำนวน: 20 แถว
💾 บันทึกไฟล์ทดสอบเรียบร้อย: iot_results.json
👉 คุณสามารถนำไฟล์นี้ไปเขียนโค้ดคำนวณ BERT Score ได้เลยครับ


In [ ]:
df.to_csv("iot_inference_results.csv", index=False)
 print("\nบันทึกผลลัพธ์ทั้งหมดไปที่ iot_inference_results.csv แล้ว")